# 03 — Decision Engine: Motor costo-beneficio

En esta fase convertimos la salida del modelo ML en una decisión operativa.

El objetivo no es que el LLM “decida por intuición”, sino construir una capa determinista y auditable que responda:

- ¿Hay riesgo de quiebre?
- ¿Cuántas unidades faltan durante el lead time?
- ¿Existe otro CEDI con excedente?
- ¿Cuánto cuesta mover inventario?
- ¿Cuánto se pierde si no actuamos?
- ¿Cuál es la recomendación final?

Esta separación es clave para entrevistas de arquitectura: **el agente explica y orquesta; las decisiones críticas se calculan con reglas verificables**.

## 1. Idea de arquitectura

El flujo de esta fase es:

```text
Dataset procesado + modelo XGBoost
        ↓
Predicción de riesgo por SKU/CEDI
        ↓
Estimación de unidades faltantes
        ↓
Evaluación de CEDIs origen
        ↓
Cálculo costo-beneficio
        ↓
Recomendación estructurada
```

¿Por qué esto importa? Porque un modelo que solo dice “riesgo alto” no resuelve el caso de negocio. El director necesita saber **qué acción tomar y por qué financieramente conviene**.

In [ ]:
from pathlib import Path
import pandas as pd
import joblib

BASE_DIR = Path("..")
DATASET_PATH = BASE_DIR / "outputs" / "herdez_features_dataset.csv"
MODEL_PATH = BASE_DIR / "models" / "best_stockout_model.joblib"

df = pd.read_csv(DATASET_PATH, parse_dates=["Fecha"])
artifact = joblib.load(MODEL_PATH)

print(df.shape)
print(artifact.keys())
print("Threshold del modelo:", artifact["threshold"])
df.head()

## 2. Scoring: convertir filas en probabilidades de riesgo

El modelo XGBoost entrega una probabilidad de riesgo de quiebre.

Esto nos permite separar dos cosas:

- **Probabilidad**: qué tan riesgoso es el SKU/CEDI.
- **Alerta**: si supera el umbral operativo definido durante entrenamiento.

Esta separación es importante porque el dashboard puede mostrar probabilidades, pero el motor operativo puede trabajar con alertas.

In [ ]:
feature_columns = artifact["feature_columns"]
pipeline = artifact["pipeline"]
threshold = artifact["threshold"]

scored = df.copy()
scored["Riesgo_Probabilidad"] = pipeline.predict_proba(scored[feature_columns])[:, 1]
scored["Alerta_Riesgo"] = (scored["Riesgo_Probabilidad"] >= threshold).astype(int)

scored[["Fecha", "SKU_ID", "CEDI", "Stock_Actual", "Ventas_Media_7d", "Lead_Time_Dias", "Riesgo_Probabilidad", "Alerta_Riesgo"]].head()

## 3. Reglas de negocio

Creamos parámetros configurables. No son verdades universales; representan políticas operativas iniciales:

| Parámetro | Significado |
|---|---|
| `safety_factor` | colchón adicional de demanda |
| `min_source_coverage_ratio` | inventario mínimo que debe conservar el CEDI origen |
| `high_risk_threshold` | umbral para llamar a un riesgo “alto” |
| `medium_risk_threshold` | umbral para llamar a un riesgo “medio” |

En entrevista puedes decir: “Estos umbrales son configurables según nivel de servicio objetivo y política logística”.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class DecisionConfig:
    safety_factor: float = 1.10
    min_source_coverage_ratio: float = 1.20
    high_risk_threshold: float = 0.70
    medium_risk_threshold: float = 0.40
    min_net_benefit: float = 0.0

config = DecisionConfig()
config

## 4. Funciones principales del motor

Estas funciones son deterministas. Eso significa que, con los mismos datos de entrada, siempre devuelven la misma salida.

Esto es importante para auditoría: si alguien pregunta por qué se recomendó mover inventario, podemos reconstruir la decisión.

In [ ]:
import numpy as np

def classify_risk_level(probability, config):
    if probability >= config.high_risk_threshold:
        return "Alto"
    if probability >= config.medium_risk_threshold:
        return "Medio"
    return "Bajo"

def estimate_demand_during_lead_time(row, config):
    return float(row["Ventas_Media_7d"] * row["Lead_Time_Dias"] * config.safety_factor)

def estimate_units_needed(row, config):
    demand_lt = estimate_demand_during_lead_time(row, config)
    needed = max(0.0, demand_lt - float(row["Stock_Actual"]))
    return int(np.ceil(needed))

def expected_stockout_loss(row):
    return float(row["Riesgo_Probabilidad"] * row["Costo_Quiebre_Stock_Diario"] * row["Lead_Time_Dias"])

def transfer_cost(row, units):
    return float(units * row["Costo_Transferencia_Unidad"])

def source_surplus(source_row, config):
    source_demand_lt = estimate_demand_during_lead_time(source_row, config)
    min_stock_required = source_demand_lt * config.min_source_coverage_ratio
    return int(max(0, np.floor(float(source_row["Stock_Actual"]) - min_stock_required)))

## 5. Ejemplo con una alerta real

Tomamos la alerta con mayor probabilidad de riesgo para explicar el cálculo paso a paso.

In [ ]:
alert = scored[scored["Alerta_Riesgo"] == 1].sort_values("Riesgo_Probabilidad", ascending=False).iloc[0]

units_needed = estimate_units_needed(alert, config)
expected_loss = expected_stockout_loss(alert)
risk_level = classify_risk_level(alert["Riesgo_Probabilidad"], config)

print("SKU:", alert["SKU_ID"])
print("CEDI destino:", alert["CEDI"])
print("Riesgo:", round(alert["Riesgo_Probabilidad"], 4), risk_level)
print("Stock actual:", alert["Stock_Actual"])
print("Ventas media 7d:", round(alert["Ventas_Media_7d"], 2))
print("Lead time:", alert["Lead_Time_Dias"])
print("Unidades necesarias:", units_needed)
print("Pérdida esperada sin actuar:", round(expected_loss, 2))

## 6. Buscar CEDIs origen

Ahora buscamos otros CEDIs que tengan el mismo SKU en la misma fecha.

Pero no basta con tener stock. Aplicamos una política:

> El CEDI origen debe conservar inventario suficiente para cubrir su propia demanda durante el lead time.

Esto evita “salvar” un CEDI provocando quiebre en otro.

In [ ]:
candidates = scored[
    (scored["Fecha"] == alert["Fecha"]) &
    (scored["SKU_ID"] == alert["SKU_ID"]) &
    (scored["CEDI"] != alert["CEDI"])
].copy()

candidates["Excedente_Transferible"] = candidates.apply(lambda r: source_surplus(r, config), axis=1)
candidates[["Fecha", "SKU_ID", "CEDI", "Stock_Actual", "Ventas_Media_7d", "Lead_Time_Dias", "Riesgo_Probabilidad", "Excedente_Transferible"]]

## 7. Cálculo costo-beneficio

Para cada CEDI origen viable calculamos:

```text
Pérdida evitada = pérdida esperada × proporción del faltante cubierto
Costo transferencia = unidades transferidas × costo por unidad
Beneficio neto = pérdida evitada - costo transferencia
```

Este ajuste es importante: si solo podemos transferir parte del faltante, no debemos asumir que eliminamos toda la pérdida.

In [ ]:
def evaluate_candidate(alert, source, config):
    needed = estimate_units_needed(alert, config)
    excess = source_surplus(source, config)
    units = min(needed, excess)
    if units <= 0:
        return None

    loss_before = expected_stockout_loss(alert)
    coverage_ratio = min(1.0, units / needed) if needed > 0 else 1.0
    loss_avoided = loss_before * coverage_ratio
    cost = transfer_cost(alert, units)
    net_benefit = loss_avoided - cost

    return {
        "CEDI_Origen": source["CEDI"],
        "Unidades_A_Transferir": units,
        "Perdida_Evitada": round(loss_avoided, 2),
        "Costo_Transferencia": round(cost, 2),
        "Beneficio_Neto": round(net_benefit, 2),
        "Riesgo_Origen": round(source["Riesgo_Probabilidad"], 4),
        "Excedente_Origen": excess,
    }

scenarios = [evaluate_candidate(alert, row, config) for _, row in candidates.iterrows()]
scenarios = [s for s in scenarios if s is not None]
pd.DataFrame(scenarios).sort_values("Beneficio_Neto", ascending=False)

## 8. Recomendación final

La recomendación no se basa solo en “riesgo alto”. Se basa en:

1. Riesgo del modelo.
2. Faltante estimado durante lead time.
3. Origen viable.
4. Beneficio neto positivo.
5. Política de no dañar al CEDI origen.

In [ ]:
scenario_df = pd.DataFrame(scenarios)

if units_needed <= 0:
    action = "MONITOR"
    reason = "El inventario actual cubre la demanda esperada."
elif scenario_df.empty:
    action = "EXPEDITE_REPLENISHMENT_OR_REVIEW"
    reason = "No hay CEDI origen con excedente suficiente."
else:
    best = scenario_df.sort_values("Beneficio_Neto", ascending=False).iloc[0]
    if best["Beneficio_Neto"] > config.min_net_benefit:
        action = "TRANSFER_INVENTORY"
        reason = f"Mover {best['Unidades_A_Transferir']} unidades desde {best['CEDI_Origen']} genera beneficio neto positivo."
    else:
        action = "WAIT_REPLENISHMENT"
        reason = "Transferir no justifica el costo logístico."

print("Acción recomendada:", action)
print("Razón:", reason)

## 9. Ejecutar el script profesional

Todo lo anterior está empaquetado en `src/03_decision_engine.py`.

Puedes ejecutarlo así desde la raíz del proyecto:

```bash
python src/03_decision_engine.py \
  --processed outputs/herdez_features_dataset.csv \
  --model models/best_stockout_model.joblib \
  --output-dir outputs \
  --max-alerts 50
```

Genera:

- `scored_stockout_alerts.csv`
- `decision_recommendations.csv`
- `decision_transfer_scenarios.csv`
- `decision_engine_report.md`
- `decision_engine_config.json`

## 10. Cómo explicarlo en entrevista

> El modelo predictivo identifica el riesgo de quiebre por SKU/CEDI. Después, un motor determinista calcula si conviene mover inventario comparando la pérdida esperada contra el costo logístico. Además, valida que el CEDI origen no quede en riesgo. El agente GenAI no inventa la decisión: solo orquesta, pregunta, explica y traduce la recomendación al lenguaje de negocio.

Esta frase cubre enfoque técnico, caso de negocio, creatividad, propuesta, caso de uso y soft skills.